# Tutorial 2: Real Data, Anisotropy, and BPX Export

In the first tutorial, we generated a synthetic gyroid microstructure. However, 
in real-world materials science, you will typically work with 3D image stacks 
acquired from X-ray Computed Tomography (XRT) or FIB-SEM.

Furthermore, manufactured porous media—like compressed battery electrodes—are 
rarely isotropic. The calendering (compression) process forces particles closer 
together in the through-thickness direction, making transport harder in that 
direction compared to the in-plane directions.

**In this tutorial we will:**
1. Load a real 3D TIFF image stack of a two-phase microstructure.
2. Compute the tortuosity in all three cardinal directions (X, Y, Z) to measure anisotropy.
3. Export the data to a BPX-compatible (Battery Parameter eXchange) JSON format, 
   ready to be dropped directly into continuum battery models like PyBaMM.

In [ ]:
# @title Setup and Imports
import os
import json
import numpy as np
import matplotlib.pyplot as plt
import tifffile

import openimpala as oi
import openimpala.core as oic

print(f"OpenImpala version: {oi.__version__}")

## 1. Loading Real Tomography Data

OpenImpala's C++ backend has built-in readers for TIFF, RAW, and HDF5. However, 
when using the Python API, the most "Pythonic" workflow is to load your data 
using standard scientific libraries like `tifffile` or `h5py`, preprocess it 
(e.g., cropping, filtering), and then hand the NumPy array to OpenImpala.

We will use the sample dataset provided in the OpenImpala repository.

In [ ]:
# Path to the sample dataset in the OpenImpala repo
data_path = "../data/SampleData_2Phase_stack_3d_1bit.tif"

if not os.path.exists(data_path):
    raise FileNotFoundError(f"Could not find {data_path}. Ensure you are running this from the tutorials/ directory.")

# Read the TIFF stack into a 3D NumPy array (Z, Y, X)
microstructure = tifffile.imread(data_path)

# OpenImpala expects 32-bit integers for phase IDs
microstructure = microstructure.astype(np.int32)

print(f"Loaded image shape (Z, Y, X): {microstructure.shape}")
print(f"Unique phases present:        {np.unique(microstructure)}")

# Visualize the middle slice
z_mid = microstructure.shape[0] // 2
plt.figure(figsize=(6, 6))
plt.imshow(microstructure[z_mid, :, :], cmap='gray')
plt.title(f"Microstructure Slice at Z = {z_mid}")
plt.axis('off')
plt.show()

## 2. Calculating Directional Tortuosity (Anisotropy)

Let's assume Phase 1 is our pore space (electrolyte). We want to know how 
easily lithium ions can diffuse through this space. 

We will compute the volume fraction, and then use a loop to calculate the 
tortuosity in the X, Y, and Z directions. Because AMReX and HYPRE initialization 
has a slight overhead, we wrap the entire loop in a single `oi.Session()` 
to keep things fast and memory-safe.

In [ ]:
directions =['x', 'y', 'z']
tau_results = {}

with oi.Session():
    # 1. Compute Volume Fraction
    vf = oi.volume_fraction(microstructure, phase=1)
    print(f"Pore Volume Fraction (Porosity): {vf.fraction:.4f}\n")
    
    # 2. Compute Tortuosity for each direction
    for d in directions:
        print(f"Solving for direction: {d.upper()}...")
        
        # Check percolation first to avoid solver errors on blocked domains
        perc = oi.percolation_check(microstructure, phase=1, direction=d)
        
        if perc.percolates:
            res = oi.tortuosity(microstructure, phase=1, direction=d, solver="flexgmres")
            tau_results[d.upper()] = res.tortuosity
            print(f"  -> Converged in {res.iterations} iterations. Tortuosity = {res.tortuosity:.4f}")
        else:
            tau_results[d.upper()] = None
            print("  -> Phase does NOT percolate in this direction.")

### Visualizing the Anisotropy

Let's plot the results to see if the material has a preferred transport direction.

In [ ]:
dirs = list(tau_results.keys())
taus = list(tau_results.values())

plt.figure(figsize=(6, 4))
plt.bar(dirs, taus, color=['#1f77b4', '#ff7f0e', '#2ca02c'])
plt.axhline(y=1.0, color='r', linestyle='--', label='Ideal limit (Tau=1.0)')
plt.ylabel("Tortuosity Factor ($\\tau$)")
plt.title("Directional Tortuosity (Anisotropy)")
plt.ylim(1.0, max(taus) * 1.1)
plt.legend()
plt.show()

## 3. Exporting to Battery Models (BPX Format)

OpenImpala is designed to bridge the gap between 3D imaging and 1D continuum 
battery models (like PyBaMM and DandeLiion). 

To do this, OpenImpala provides a native `ResultsJSON` builder in its low-level 
C++ core. This builder generates files compliant with the **Battery Parameter eXchange (BPX)** standard.

*Note: In physics definitions, effective diffusivity $D_{eff} = D_{bulk} \\frac{\\varepsilon}{\\tau}$.*
*The ratio $\\frac{D_{eff}}{D_{bulk}}$ is also known as the **Transport Efficiency**.*

In [ ]:
# Access the low-level C++ bindings via openimpala.core
rj = oic.ResultsJSON()

# Define the physics we just solved (Diffusion)
physics_cfg = oic.PhysicsConfig.from_type_string("diffusion")
rj.set_physics_config(physics_cfg)

# Set metadata
rj.set_input_file(data_path)
rj.set_phase_id(1)
rj.set_volume_fraction(vf.fraction)

# Add our directional results
for d, tau in tau_results.items():
    if tau is not None:
        # OpenImpala's JSON builder expects the Transport Efficiency (D_eff_ratio)
        # transport_efficiency = porosity / tortuosity
        transport_efficiency = vf.fraction / tau
        rj.add_direction_result(d, transport_efficiency)

# Enable BPX fragment generation for a specific electrode (e.g., the positive cathode)
rj.set_bpx_electrode("positive")
rj.set_provenance(sample_id="SampleData_2Phase", provenance_uri="https://github.com/BASE-Laboratory/OpenImpala")

# Build the JSON string
bpx_json_string = rj.build_json_string()

# Print the resulting JSON
print("Generated BPX-Compatible JSON output:")
print("-" * 50)
print(bpx_json_string)
print("-" * 50)

### Next Steps

You can now save this JSON to disk using `rj.write("my_electrode.json")`. 
The resulting `bpx` block can be pasted directly into a full BPX parameter 
file and loaded into PyBaMM to simulate a battery cell using the exact 
transport properties measured from your 3D image!